# Feature Engineering Phase 1

This experiment tests a small set of target-independent, interpretable features against the existing LightGBM baseline. Every model uses the same fixed hyperparameters and chronological validation set, isolating feature impact from tuning.

**Guardrails:** frequency maps and relative-time origin are fitted on training only. Validation is transform-only. Holdout is never assigned or evaluated. No target encoding, behavioral velocity, graph features, threshold optimization, or SHAP is used.

In [1]:
import gc
import sys
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
    validate_training_data,
)
from fraud_detection.features import Phase1FeatureEngineer
from fraud_detection.models import TrainingOnlyPreprocessor, evaluate_probabilities

plt.style.use("seaborn-v0_8-whitegrid")

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


2026/10/06 06:25:54 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\fraud-detection-system\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Chronological train/validation design

The split is recreated using project utilities. Structural validation is run only on the selected training partition in this notebook.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validation = split.validation
validate_training_data(train)
del transaction, identity, merged
gc.collect()

excluded_columns = ["isFraud", "TransactionID"]
feature_columns = [column for column in train.columns if column not in excluded_columns]
train_predictors = train[feature_columns]
validation_predictors = validation[feature_columns]
y_train = train["isFraud"].astype("int8")
y_validation = validation["isFraud"].astype("int8")

print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Baseline feature count: {len(feature_columns)}")
print("Holdout status: reserved and not referenced by this experiment")

Train rows: 413,378
Validation rows: 88,581
Baseline feature count: 432
Holdout status: reserved and not referenced by this experiment


## 2. Phase 1 groups

- **Amount/time:** `log_transaction_amount`, train-origin `elapsed_day`, and repeating `relative_hour`. `TransactionDT` remains raw; its calendar origin and timezone are unknown.
- **Missing indicators:** selected device, email, and distance fields found during training-only EDA.
- **Frequency encodings:** selected card, address, device, and email values. Maps use no target and are fitted only on training; unseen validation values map to zero.

The full ablation includes baseline, each group independently, and all groups together.

In [3]:
feature_sets = {
    "baseline": None,
    "amount_time": dict(
        include_amount_time=True,
        include_missing_indicators=False,
        include_frequency_encodings=False,
    ),
    "missing_indicators": dict(
        include_amount_time=False,
        include_missing_indicators=True,
        include_frequency_encodings=False,
    ),
    "frequency_encodings": dict(
        include_amount_time=False,
        include_missing_indicators=False,
        include_frequency_encodings=True,
    ),
    "all_phase1": dict(
        include_amount_time=True,
        include_missing_indicators=True,
        include_frequency_encodings=True,
    ),
}

lightgbm_parameters = dict(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-feature-engineering")
feature_sets

2026/10/06 06:26:18 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-feature-engineering' does not exist. Creating a new experiment.


{'baseline': None,
 'amount_time': {'include_amount_time': True,
  'include_missing_indicators': False,
  'include_frequency_encodings': False},
 'missing_indicators': {'include_amount_time': False,
  'include_missing_indicators': True,
  'include_frequency_encodings': False},
 'frequency_encodings': {'include_amount_time': False,
  'include_missing_indicators': False,
  'include_frequency_encodings': True},
 'all_phase1': {'include_amount_time': True,
  'include_missing_indicators': True,
  'include_frequency_encodings': True}}

## 3. Fixed-parameter ablation

Each iteration fits feature transformations and baseline preprocessing from training only, then scores one fixed LightGBM on validation.

In [4]:
rows = []
for feature_set_name, feature_options in feature_sets.items():
    if feature_options is None:
        engineered_train = train_predictors
        engineered_validation = validation_predictors
        generated_features = []
    else:
        feature_engineer = Phase1FeatureEngineer(**feature_options).fit(train_predictors)
        engineered_train = feature_engineer.transform(train_predictors)
        engineered_validation = feature_engineer.transform(validation_predictors)
        generated_features = list(feature_engineer.generated_feature_names_)

    preprocessor = TrainingOnlyPreprocessor()
    x_train = preprocessor.fit_transform(engineered_train)
    x_validation = preprocessor.transform(engineered_validation)
    scaler = StandardScaler(copy=False)
    x_train = scaler.fit_transform(x_train)
    x_validation = scaler.transform(x_validation)

    with mlflow.start_run(run_name=f"phase1-{feature_set_name}") as run:
        model = lgb.LGBMClassifier(**lightgbm_parameters)
        model.fit(x_train, y_train)
        probabilities = model.predict_proba(x_validation)[:, 1]
        metrics = evaluate_probabilities(y_validation, probabilities)
        mlflow.set_tags(
            {
                "stage": "feature-engineering",
                "dataset": "ieee-cis",
                "split": "temporal-validation",
            }
        )
        mlflow.log_params(
            {
                "feature_set_name": feature_set_name,
                "feature_count": x_train.shape[1],
                "learning_rate": lightgbm_parameters["learning_rate"],
                "n_estimators": lightgbm_parameters["n_estimators"],
                "num_leaves": lightgbm_parameters["num_leaves"],
            }
        )
        mlflow.log_metrics(
            {
                "pr_auc": metrics.pr_auc,
                "roc_auc": metrics.roc_auc,
                "precision_at_0_50": metrics.precision,
                "recall_at_0_50": metrics.recall,
                "f1_at_0_50": metrics.f1,
            }
        )
        rows.append(
            {
                "Version": feature_set_name,
                "Feature Count": x_train.shape[1],
                "PR-AUC": metrics.pr_auc,
                "ROC-AUC": metrics.roc_auc,
                "Precision": metrics.precision,
                "Recall": metrics.recall,
                "F1": metrics.f1,
                "Generated Features": generated_features,
                "MLflow Run ID": run.info.run_id,
            }
        )

    del x_train, x_validation, model, probabilities
    if feature_options is not None:
        del engineered_train, engineered_validation, feature_engineer
    gc.collect()

results = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
display(results.drop(columns="Generated Features"))

,Version,Feature Count,PR-AUC,ROC-AUC,Precision,Recall,F1,MLflow Run ID
0,frequency_encodings,438,0.589631,0.921839,0.577205,0.544379,0.560311,267c59c9fa2b438e988751faefbd06aa
1,baseline,432,0.587317,0.921482,0.572638,0.533859,0.552569,11ce177577324f348508da0dc8dc3d2d
2,missing_indicators,437,0.578732,0.916614,0.559520,0.536160,0.547591,3a1aaac1a2a7420795bdee49e32f5cf3
3,all_phase1,446,0.574197,0.921037,0.513078,0.554569,0.533017,e42dd1e0c0994a4c8c8ee6bc2527e2fa
4,amount_time,435,0.573694,0.918147,0.503237,0.562130,0.531056,8a6422fa41a84412aa1cfdaf1f516adb


## 4. Baseline versus all Phase 1 features

In [5]:
baseline = results.loc[results["Version"] == "baseline"].iloc[0]
engineered = results.loc[results["Version"] == "all_phase1"].iloc[0]
absolute_improvement = engineered["PR-AUC"] - baseline["PR-AUC"]
relative_improvement = absolute_improvement / baseline["PR-AUC"]
comparison = pd.DataFrame([baseline, engineered]).drop(
    columns=["Generated Features", "MLflow Run ID"]
)
display(comparison)
print(f"Absolute PR-AUC change: {absolute_improvement:+.6f}")
print(f"Relative PR-AUC change: {relative_improvement:+.2%}")

,Version,Feature Count,PR-AUC,ROC-AUC,Precision,Recall,F1
1,baseline,432,0.587317,0.921482,0.572638,0.533859,0.552569
3,all_phase1,446,0.574197,0.921037,0.513078,0.554569,0.533017


Absolute PR-AUC change: -0.013120
Relative PR-AUC change: -2.23%


In [6]:
figure_path = PROJECT_ROOT / "reports" / "figures" / "feature_engineering_phase1_ablation.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
plot_data = results.sort_values("PR-AUC")
fig, axis = plt.subplots(figsize=(9, 5))
axis.barh(plot_data["Version"], plot_data["PR-AUC"], color="#4C78A8")
axis.axvline(baseline["PR-AUC"], color="#E45756", linestyle="--", label="Baseline")
axis.set(title="Phase 1 feature-group ablation", xlabel="Validation PR-AUC", ylabel="Feature set")
axis.legend()
fig.tight_layout()
fig.savefig(figure_path, dpi=160)
plt.close(fig)


def markdown_results_table(frame):
    header = "| Version | Features | PR-AUC | ROC-AUC | Precision | Recall | F1 |"
    divider = "| --- | ---: | ---: | ---: | ---: | ---: | ---: |"
    table_rows = []
    for _, row in frame.iterrows():
        table_rows.append(
            f"| {row['Version']} | {int(row['Feature Count'])} | "
            f"{row['PR-AUC']:.6f} | {row['ROC-AUC']:.6f} | "
            f"{row['Precision']:.6f} | {row['Recall']:.6f} | "
            f"{row['F1']:.6f} |"
        )
    return "\n".join([header, divider, *table_rows])


best = results.iloc[0]
result_word = "improvement" if absolute_improvement > 0 else "degradation"
report = f"""# Feature Engineering Phase 1

## Experimental design

All experiments fit on the chronological training split and score the fixed validation split.
The holdout set is not assigned or evaluated. LightGBM parameters are fixed at
`learning_rate=0.1`, `n_estimators=400`, and `num_leaves=63`.

Frequency maps, numeric preprocessing statistics, and the relative-time origin are learned from
training only. Unseen validation categories receive frequency zero. No target-derived feature is
used.

## Observed results

{markdown_results_table(results)}

- Baseline features: **{int(baseline["Feature Count"])}**
- All Phase 1 features: **{int(engineered["Feature Count"])}**
- Absolute PR-AUC {result_word}: **{absolute_improvement:+.6f}**
- Relative PR-AUC {result_word}: **{relative_improvement:+.2%}**
- Best observed feature set: **{best["Version"]}** ({best["PR-AUC"]:.6f} PR-AUC)

![Phase 1 ablation](figures/feature_engineering_phase1_ablation.png)

## Interpretation

These are validation associations under fixed model parameters. A group is helpful only if its
PR-AUC exceeds the reproduced baseline. Features that do not improve ranking are not claimed as
wins.

## Limitations

- The existing baseline preprocessor already frequency-encodes object columns, so explicit frequency
  features for object-valued candidates can be redundant; numeric entity IDs gain the clearest new
  view.
- No hyperparameter retuning, calibration, threshold optimization, or holdout evaluation is
  performed.
- Relative hour is based on an origin-unknown counter, not a known calendar timezone.

## Next proposed phase (not implemented)

Evaluate leakage-safe, past-only behavioral velocity and entity-history aggregates before graph
features.
"""
report_path = PROJECT_ROOT / "reports" / "feature_engineering_phase1_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Figure: {figure_path.relative_to(PROJECT_ROOT)}")

Report: reports\feature_engineering_phase1_summary.md
Figure: reports\figures\feature_engineering_phase1_ablation.png


## 5. Scope boundary

This notebook stops after Phase 1 impact measurement. It does not implement velocity, rolling behavior, graph features, rules, SHAP, threshold optimization, retraining, serving, or deployment.